In [1]:
import sklearn
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
sklearn_version = sklearn.__version__

sklearn_version

'1.0.2'

In [8]:
sentiment_data = pd.read_csv('datasets/sentimental_analysis_data.csv', header=None, names=['Label', 'Text'], sep='\t')

sentiment_data.sample(10)

,Label,Text
5782,0,I hate Harry Potter.
4152,0,"by the way, the Da Vinci Code sucked, just let..."
4851,0,"Da Vinci Code = Up, Up, Down, Down, Left, Righ..."
5624,0,Harry Potter dragged Draco Malfoy ’ s trousers...
5868,0,"I hate Harry Potter, it's retarted, gay and st..."
4140,0,The Da Vinci Code sucked big time.
1187,1,What I'd like to see is some Mission Impossibl...
2992,1,and i wanna shout out a big fat thank you to e...
5437,0,This quiz sucks and Harry Potter sucks ok bye..
6077,0,"After the festivities, madre and I saw Brokeba..."


In [5]:
sentiment_data.shape

(6918, 2)

In [9]:
X = sentiment_data['Text']

Y = sentiment_data['Label']

In [10]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2)

In [11]:
tfidf_vect = TfidfVectorizer(max_features=15)

x_trans = tfidf_vect.fit_transform(x_train)

In [12]:
print(x_trans[0:3])

  (0, 6)	0.7071067811865475
  (0, 9)	0.7071067811865475
  (1, 3)	0.448065129086165
  (1, 13)	0.4481905611172774
  (1, 4)	0.4481905611172774
  (1, 12)	0.39759759771694553
  (1, 8)	0.48928951780870206
  (2, 10)	0.5629633029116565
  (2, 2)	0.5629633029116565
  (2, 0)	0.605098867251953


In [14]:
x_trans.shape

(5534, 15)

In [16]:
classifier = LinearSVC(C=1.0, max_iter=1000, tol=1e-3)
linear_svc_model = classifier.fit(x_trans, y_train)

linear_svc_model

LinearSVC(tol=0.001)

In [18]:
x_test_trans = tfidf_vect.fit_transform(x_test)

x_test_trans.shape

(1384, 15)

In [19]:
y_pred = linear_svc_model.predict(x_test_trans)

y_pred

array([0, 1, 1, ..., 0, 0, 1])

In [20]:
pred_results = pd.DataFrame({'y_test': y_test, 'y_pred': y_pred})

pred_results.sample(5)

,y_test,y_pred
3709,1,0
3798,1,0
524,1,1
1385,1,1
5737,0,0


In [21]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

accuracy

0.9082369942196532

In [27]:
text_clf_param = {}

text_clf_param['preprocessing'] = tfidf_vect
text_clf_param['model'] = linear_svc_model
text_clf_param['sklearn_version'] = sklearn_version
text_clf_param['accuracy'] = accuracy

In [28]:
text_clf_param

{'preprocessing': TfidfVectorizer(max_features=15),
 'model': LinearSVC(tol=0.001),
 'sklearn_version': '1.0.2',
 'accuracy': 0.9082369942196532}

In [24]:
import joblib

In [25]:
filename = 'models/text_clf_checkpoint.joblib'

In [29]:
joblib.dump(text_clf_param, filename)

['models/text_clf_checkpoint.joblib']

In [30]:
clf_checkpoint = joblib.load(filename)

In [31]:
reloaded_vect = clf_checkpoint['preprocessing']

reloaded_vect

TfidfVectorizer(max_features=15)

In [32]:
clf_model = clf_checkpoint['model']

clf_model

LinearSVC(tol=0.001)

In [33]:
x_test_trans_new = reloaded_vect.fit_transform(x_test)

In [34]:
y_pred = linear_svc_model.predict(x_test_trans_new)

y_pred

array([0, 1, 1, ..., 0, 0, 1])

In [36]:
accuracy = accuracy_score(y_test, y_pred)

accuracy

0.9082369942196532

In [37]:
from sklearn.pipeline import Pipeline

In [38]:
clf_pipeline = Pipeline(steps=[('tfidf_vect', tfidf_vect), ('classifier', classifier)])

pipeline_model = clf_pipeline.fit(x_train, y_train)

In [39]:
y_pred = pipeline_model.predict(x_test)

In [47]:
accuracy = accuracy_score(y_test, y_pred)

accuracy

0.9082369942196532

In [48]:
pipe_clf_param = {}

pipe_clf_param['pipeline_clf'] = pipeline_model
pipe_clf_param['sklearn_version'] = sklearn_version
pipe_clf_param['accuracy'] = accuracy

In [49]:
filename = 'models/pipe_clf_checkpoint.joblib'

In [50]:
joblib.dump(pipe_clf_param, filename)

['models/pipe_clf_checkpoint.joblib']

In [51]:
pipe_clf_checkpoint = joblib.load(filename)

In [52]:
reloaded_pipeline = pipe_clf_checkpoint['pipeline_clf']

reloaded_pipeline

Pipeline(steps=[('tfidf_vect', TfidfVectorizer(max_features=15)),
                ('classifier', LinearSVC(tol=0.001))])

In [53]:
y_pred = reloaded_pipeline.predict(x_test)

In [54]:
accuracy = accuracy_score(y_test, y_pred)

accuracy

0.9082369942196532